In [ ]:
# === Colab installs ===
# You can comment this cell out if your runtime already has these.
import sys, subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Core deps for this notebook
try:
    import torch, torchvision  # noqa: F401
except Exception:
    pip_install("torch", "torchvision", "torchaudio")

# Sanity check: show torch/torchvision versions and CUDA availability
import torch, torchvision
print("Torch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


: 

In [ ]:
# === Imports & setup ===
import math
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms

SEED = 1337
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# === ThresholdReLU (custom autograd) ===
class ThresholdReLUFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, t):
        if not torch.is_tensor(t):
            t = torch.tensor(t, dtype=x.dtype, device=x.device)
        else:
            t = t.to(dtype=x.dtype, device=x.device)
        ctx.save_for_backward(x, t)
        return torch.where(x > t, x, torch.zeros_like(x))

    @staticmethod
    def backward(ctx, grad_out):
        x, t = ctx.saved_tensors
        mask = (x > t).to(grad_out.dtype)
        grad_x = grad_out * mask
        # t is not learned (non-differentiable here)
        return grad_x, None

class ThresholdReLU(nn.Module):
    def forward(self, x, threshold):
        return ThresholdReLUFn.apply(x, threshold)

# Quick unit test
_act = ThresholdReLU()
_x = torch.tensor([[-1.0, 0.0, 0.5, 2.0, 7.0, 5.5, 0.9]], requires_grad=True)
_t = 3.0
_y = _act(_x, _t)
_y.sum().backward()
print("ThresholdReLU quick test OK:", _x.grad is not None)


In [ ]:
# === Datasets & loaders (flattened for MLPs) ===
class Flatten:
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        return t.view(-1)

def get_loaders(
    name="MNIST",
    data_dir="./data",
    batch_train=128,
    batch_val=256,
    val_frac=0.2,
    shuffle_train=True,
):
    """
    Returns (train_loader, val_loader, test_loader, num_classes, input_shape)
    Each sample is flattened to 1D for MLPs.
    """
    name = name.upper()
    FlattenT = Flatten()

    if name in ["MNIST", "FASHIONMNIST"]:
        if name == "MNIST":
            mean, std = (0.1307,), (0.3081,)
            DatasetClass = datasets.MNIST
        else:
            mean, std = (0.2860,), (0.3530,)
            DatasetClass = datasets.FashionMNIST

        T_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            FlattenT,
        ])
        T_eval = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            FlattenT,
        ])

        full_train_aug  = DatasetClass(root=data_dir, train=True,  download=True, transform=T_train)
        full_train_eval = DatasetClass(root=data_dir, train=True,  download=True, transform=T_eval)
        test_ds         = DatasetClass(root=data_dir, train=False, download=True, transform=T_eval)

        num_classes, input_shape = 10, (1 * 28 * 28,)

        # Match val split across augmented/eval copies using shared indices
        n_val = int(len(full_train_aug) * val_frac)
        n_tr  = len(full_train_aug) - n_val
        gen = torch.Generator().manual_seed(SEED)
        idx_train, idx_val = torch.utils.data.random_split(range(len(full_train_aug)), [n_tr, n_val], generator=gen)

        train_ds = torch.utils.data.Subset(full_train_aug,  idx_train.indices)
        val_ds   = torch.utils.data.Subset(full_train_eval, idx_val.indices)

        train_loader = DataLoader(train_ds, batch_size=batch_train, shuffle=shuffle_train, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        return train_loader, val_loader, test_loader, num_classes, input_shape

    elif name in ["CIFAR10", "CIFAR-10"]:
        mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        T_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            FlattenT,
        ])
        T_eval = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            FlattenT,
        ])

        full_train_aug  = datasets.CIFAR10(root=data_dir, train=True,  download=True, transform=T_train)
        full_train_eval = datasets.CIFAR10(root=data_dir, train=True,  download=True, transform=T_eval)
        test_ds         = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=T_eval)

        num_classes, input_shape = 10, (3 * 32 * 32,)

        n_val = int(len(full_train_aug) * val_frac)
        n_tr  = len(full_train_aug) - n_val
        gen = torch.Generator().manual_seed(SEED)
        idx_train, idx_val = torch.utils.data.random_split(range(len(full_train_aug)), [n_tr, n_val], generator=gen)

        train_ds = torch.utils.data.Subset(full_train_aug,  idx_train.indices)
        val_ds   = torch.utils.data.Subset(full_train_eval, idx_val.indices)

        train_loader = DataLoader(train_ds, batch_size=batch_train, shuffle=shuffle_train, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        return train_loader, val_loader, test_loader, num_classes, input_shape

    elif name == "SVHN":
        mean, std = (0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)
        T = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            FlattenT,
        ])
        full_train = datasets.SVHN(root=data_dir, split='train', download=True, transform=T)
        test_ds    = datasets.SVHN(root=data_dir, split='test',  download=True, transform=T)
        num_classes, input_shape = 10, (3 * 32 * 32,)

        n_val = int(len(full_train) * val_frac)
        n_tr  = len(full_train) - n_val
        gen = torch.Generator().manual_seed(SEED)
        train_ds, val_ds = random_split(full_train, [n_tr, n_val], generator=gen)

        train_loader = DataLoader(train_ds, batch_size=batch_train, shuffle=shuffle_train, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=batch_val,   shuffle=False,        num_workers=2, pin_memory=True)
        return train_loader, val_loader, test_loader, num_classes, input_shape

    else:
        raise ValueError("Supported datasets: MNIST, FashionMNIST, CIFAR10, SVHN")

# Example: choose your dataset here
train_loader, val_loader, test_loader, num_classes, input_shape = get_loaders("MNIST")
print("Input dim:", input_shape, "| Num classes:", num_classes)


In [ ]:
# === Models ===
class AdaptiveThresholdMLP(nn.Module):
    """
    MLP with ThresholdReLU for each hidden layer.
    thresholds: list of floats, e.g., [3.0, 2.0, 1.0]
    """
    def __init__(self, input_dim, hidden_dims, output_dim, thresholds):
        super().__init__()
        assert len(hidden_dims) == len(thresholds), "len(hidden_dims) must equal len(thresholds)"
        self.thresholds = thresholds
        self.act = ThresholdReLU()

        dims = [input_dim] + hidden_dims
        self.hidden = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(hidden_dims))])
        self.out = nn.Linear(dims[-1], output_dim)

        for layer in self.hidden:
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
        nn.init.kaiming_normal_(self.out.weight, nonlinearity="linear")
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        for layer, thr in zip(self.hidden, self.thresholds):
            x = layer(x)
            x = self.act(x, thr)
        return self.out(x)

class ReLUMlp(nn.Module):
    """Same architecture, standard ReLU."""
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        dims = [input_dim] + hidden_dims
        self.hidden = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(hidden_dims))])
        self.out = nn.Linear(dims[-1], output_dim)
        self.act = nn.ReLU(inplace=False)

        for layer in self.hidden:
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
        nn.init.kaiming_normal_(self.out.weight, nonlinearity="linear")
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        for layer in self.hidden:
            x = self.act(layer(x))
        return self.out(x)

class LeakyReLUMlp(nn.Module):
    """Same architecture, LeakyReLU."""
    def __init__(self, input_dim, hidden_dims, output_dim, negative_slope: float = 0.01):
        super().__init__()
        self.negative_slope = float(negative_slope)
        dims = [input_dim] + hidden_dims
        self.hidden = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(hidden_dims))])
        self.out = nn.Linear(dims[-1], output_dim)
        self.act = nn.LeakyReLU(negative_slope=self.negative_slope, inplace=False)

        for layer in self.hidden:
            nn.init.kaiming_normal_(layer.weight, nonlinearity="leaky_relu", a=self.negative_slope)
            nn.init.zeros_(layer.bias)
        nn.init.kaiming_normal_(self.out.weight, nonlinearity="linear")
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        for layer in self.hidden:
            x = self.act(layer(x))
        return self.out(x)


In [ ]:
# === Threshold schedules ===
def make_threshold_schedule(
    num_layers: int,
    kind: str,
    scale: float = 10.0,
    *,
    alpha: float = 1.5,     # Pareto shape
    lam: float = 0.3,       # Exponential rate
    sigma: float = 0.35,    # Normal width
    mode: str = "decreasing"
) -> List[float]:
    """
    Produce a list of length `num_layers` with per-layer thresholds.
    threshold_i = scale * weight_i, where weights are normalized to [0,1].

    kind in {'pareto','exponential','normal','uniform'}.
    """
    assert num_layers >= 1, "num_layers must be >= 1"
    kind = kind.lower()
    if num_layers == 1:
        return [scale * 1.0]

    L = num_layers
    idx = list(range(L))
    u = [i / (L - 1) for i in idx]  # normalized depth

    if kind in ("pareto", "powerlaw", "power"):
        w = [1.0 / ((i + 1) ** alpha) for i in idx]
        m = max(w); w = [wi / m for wi in w]

    elif kind in ("exponential", "exp"):
        w = [math.exp(-lam * i) for i in idx]
        m = max(w); w = [wi / m for wi in w]

    elif kind in ("normal", "gaussian"):
        if mode == "decreasing":
            w = [math.exp(-0.5 * (ui / max(sigma, 1e-8))**2) for ui in u]
        elif mode == "centered":
            w = [math.exp(-0.5 * ((ui - 0.5) / max(sigma, 1e-8))**2) for ui in u]
        else:
            raise ValueError("normal: mode must be 'decreasing' or 'centered'")
        m = max(w); w = [wi / m for wi in w]

    elif kind in ("uniform", "const", "constant"):
        w = [1.0] * L

    else:
        raise ValueError("Unknown kind. Use 'pareto', 'exponential', 'normal', or 'uniform'.")

    thresholds = [scale * wi for wi in w]
    return thresholds

# Demo schedules (4 hidden layers)
thr_pareto   = make_threshold_schedule(4, "pareto",      scale=4, alpha=1.9)
thr_exp      = make_threshold_schedule(4, "exponential", scale=4, lam=0.7)
thr_normal   = make_threshold_schedule(4, "normal",      scale=4, sigma=0.35, mode="decreasing")
thr_uniform  = make_threshold_schedule(4, "uniform",     scale=4)

print("Pareto:   ", [round(v, 3) for v in thr_pareto])
print("Exponential:", [round(v, 3) for v in thr_exp])
print("Normal:   ", [round(v, 3) for v in thr_normal])
print("Uniform:  ", [round(v, 3) for v in thr_uniform])


In [ ]:
# === Training / eval helpers ===
loss_fn = nn.MSELoss()

def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            total += loss.item() * xb.size(0)
            count += xb.size(0)
    return total / max(count, 1)

def train(model, loader, epochs=20, lr=1e-3, tag="model"):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        if ep % 10 == 0 or ep == 1:
            tr = evaluate(model, train_loader)
            va = evaluate(model, val_loader)
            print(f"[{tag}] epoch {ep:03d} | train MSE: {tr:.4f} | val MSE: {va:.4f}")
    return model


In [ ]:
# === Train ===
print("Training ReLU model…")
relu_model = train(relu_model, train_loader, epochs=20, lr=1e-3, tag="ReLU")

print("\nTraining ThresholdReLU model…")
thr_model  = train(thr_model,  train_loader, epochs=20, lr=1e-3, tag=f"ThresholdReLU{[round(t,2) for t in thr_pareto]}")

print("\nTraining LeakyReLU model…")
leaky_model = train(leaky_model, train_loader, epochs=20, lr=1e-3, tag="LeakyReLU[0.01]")


In [ ]:
# === Compare on validation ===
thr_val   = evaluate(thr_model,   val_loader)
relu_val  = evaluate(relu_model,  val_loader)
leaky_val = evaluate(leaky_model, val_loader)

print("\nFinal validation MSE:")
print(f"  ThresholdReLU{[round(t,2) for t in thr_pareto]}: {thr_val:.4f}")
print(f"  ReLU:                              {relu_val:.4f}")
print(f"  LeakyReLU[0.01]:                   {leaky_val:.4f}")

# Sample forward pass (random vectors with the correct flat dim)
thr_model.eval(); relu_model.eval(); leaky_model.eval()
with torch.no_grad():
    xb = torch.randn(3, input_dim, device=device)
    print("\nSample preds (ThresholdReLU):", thr_model(xb).squeeze().tolist())
    print("Sample preds (ReLU):         ", relu_model(xb).squeeze().tolist())
    print("Sample preds (LeakyReLU):    ", leaky_model(xb).squeeze().tolist())
